# Task 5: does fitting on a cleaned normal core match `density_auto`?

This tests the user's hypothesis that unusual or incomplete rows were excluded **from fitting** before an unsupervised detector ranked all accounts. Cleaning uses only numeric Task 5 features on a fixed 12,000-row development sample; teammate predictions enter only the final comparison. Every candidate selects the same 2,046 alerts. Agreement does not establish anomaly accuracy or the teammate's exact method.

In [1]:
from pathlib import Path
import sys, json, joblib
ROOT=next(p for p in (Path.cwd(),*Path.cwd().parents) if (p/"src/pipeline/save_model.py").exists())
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
SRC=ROOT/"src"
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.mixture import GaussianMixture
from sklearn.model_selection import train_test_split
from sklearn.svm import OneClassSVM
from src.pipeline.random_seed import set_seed
from src.task5.anomaly_models import BinaryAnomalyModel,EmpiricalTail
from src.task5.density_auto import DensityAutoDetector
SEED=set_seed()
frames={}
for name in ["eda_original_v2","eda_1_v2"]:
    card=json.loads((SRC/"task5/data/eda"/f"{name}.json").read_text(encoding="utf-8"))
    frame=pd.read_parquet(SRC/card["splits"]["test"]["path"])
    assert frame.player_id.is_unique and "is_anomaly" not in frame
    frames[name]=frame.drop(columns="player_id")
N=len(frames["eda_original_v2"])
assert N==25889 and len(frames["eda_1_v2"])==N
dev_idx,_=train_test_split(np.arange(N),train_size=12000,test_size=4000,random_state=SEED)
scores={};fit_rows={}
def record(label,model,X,train_indices):
    model.fit(X.iloc[train_indices]);values=np.asarray(model.anomaly_score(X),dtype=float)
    assert values.shape==(N,) and np.isfinite(values).all()
    scores[label]=values;fit_rows[label]=len(train_indices)
    print(label,"fit rows",len(train_indices))
def retain_smallest(values,fraction):
    count=round(len(values)*fraction)
    return np.sort(dev_idx[np.lexsort((np.arange(len(values)),np.asarray(values)))[:count]])

In [2]:
# Cleaning scores are fitted on development rows only.
cleaning={}
for key,X in frames.items():
    tail=BinaryAnomalyModel(EmpiricalTail(top_k=8),anomaly_fraction=.08).fit(X.iloc[dev_idx])
    iforest=BinaryAnomalyModel(IsolationForest(n_estimators=160,max_samples=256,
                                               random_state=SEED,n_jobs=-1),anomaly_fraction=.08).fit(X.iloc[dev_idx])
    missing=X.iloc[dev_idx].isna().sum(axis=1).to_numpy()
    cleaning[key]={
        "tail90":retain_smallest(tail.anomaly_score(X.iloc[dev_idx]),.9),
        "tail80":retain_smallest(tail.anomaly_score(X.iloc[dev_idx]),.8),
        "iforest90":retain_smallest(iforest.anomaly_score(X.iloc[dev_idx]),.9),
        "complete90":retain_smallest(missing,.9),
    }
    for method,indices in cleaning[key].items():print(key,method,len(indices))

eda_original_v2 tail90 10800
eda_original_v2 tail80 9600
eda_original_v2 iforest90 10800
eda_original_v2 complete90 10800


eda_1_v2 tail90 10800
eda_1_v2 tail80 9600
eda_1_v2 iforest90 10800
eda_1_v2 complete90 10800


In [3]:
# Predeclared detector families and trims. The threshold is set later to 2,046 alerts for every score.
for key,X in frames.items():
    tag="original" if key=="eda_original_v2" else "engineered"
    for clean in ["tail90","tail80","iforest90","complete90"]:
        model=BinaryAnomalyModel(OneClassSVM(kernel="rbf",nu=.10,gamma="scale",cache_size=500),
                                 anomaly_fraction=.08)
        record(f"{tag}_ocsvm_{clean}",model,X,cleaning[key][clean])
    gmm=BinaryAnomalyModel(GaussianMixture(n_components=4,covariance_type="diag",reg_covar=1e-4,
                                            max_iter=120,random_state=SEED),anomaly_fraction=.08)
    record(f"{tag}_gmm_tail90",gmm,X,cleaning[key]["tail90"])
    auto=DensityAutoDetector(anomaly_fraction=.08,random_state=SEED,max_iter=40)
    record(f"{tag}_density_auto_tail90",auto,X,cleaning[key]["tail90"])
# Saved full-development OCSVM references.
for key,model_name in [("eda_original_v2","t5_r2_ocsvm_nu10_original_v2"),
                       ("eda_1_v2","t5_r2_ocsvm_nu10_engineered_v2")]:
    card=json.loads((SRC/"task5/save"/f"{model_name}.json").read_text(encoding="utf-8"))
    model=joblib.load(SRC/card["model_path"])
    scores[model_name]=model.anomaly_score(frames[key]);fit_rows[model_name]=12000

original_ocsvm_tail90 fit rows 10800


original_ocsvm_tail80 fit rows 9600


original_ocsvm_iforest90 fit rows 10800


original_ocsvm_complete90 fit rows 10800


original_gmm_tail90 fit rows 10800


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


... (4 outputs left out)

engineered_ocsvm_complete90 fit rows 10800


engineered_gmm_tail90 fit rows 10800


C:\Users\BTCOM\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (40) reached and the optimization hasn't converged yet.
  warnings.warn(


engineered_density_auto_tail90 fit rows 10800


In [4]:
# Teammate answers are read only after every score is fitted.
FRIEND=Path.home()/"Downloads"/"kaggle_t12345_t1_slow800_bagging_v4_t3_conditional_quantile_v1_t4_resnet18_cosine18_tta_v1_t5_density_auto_v1.csv"
friend=pd.read_csv(FRIEND);sample=pd.read_csv(SRC/"submission/sample_submission.csv",usecols=["id"])
assert len(friend)==N and friend.id.equals(sample.id) and set(friend.task5)=={0,1}
y=friend.task5.to_numpy(dtype=np.int8);k=int(y.sum());assert k==2046
rows=[]
for label,values in scores.items():
    selected=np.lexsort((np.arange(N),-values))[:k];common=int(y[selected].sum())
    rows.append({"candidate":label,"fit_rows":fit_rows[label],"matching_alerts":common,
                 "friend_coverage":common/k,"positive_jaccard":common/(2*k-common)})
results=pd.DataFrame(rows).sort_values("matching_alerts",ascending=False).reset_index(drop=True)
display(results)
print("Best",results.iloc[0].to_dict())

,candidate,fit_rows,matching_alerts,friend_coverage,positive_jaccard
0,original_ocsvm_tail80,9600,1347,0.658358,0.490710
1,original_ocsvm_tail90,10800,1337,0.653470,0.485299
2,original_ocsvm_iforest90,10800,1327,0.648583,0.479928
3,original_ocsvm_complete90,10800,1316,0.643206,0.474063
4,t5_r2_ocsvm_nu10_original_v2,12000,1313,0.641740,0.472472
5,engineered_gmm_tail90,10800,1226,0.599218,0.427774
6,engineered_ocsvm_tail80,9600,1225,0.598729,0.427276
7,engineered_ocsvm_tail90,10800,1221,0.596774,0.425287
8,engineered_ocsvm_complete90,10800,1214,0.593353,0.421821
9,engineered_ocsvm_iforest90,10800,1213,0.592864,0.421327


Best {'candidate': 'original_ocsvm_tail80', 'fit_rows': 9600, 'matching_alerts': 1347, 'friend_coverage': 0.658357771260997, 'positive_jaccard': 0.4907103825136612}


Only the fit sample changes within each cleaned OCSVM comparison. Selecting the best method after seeing teammate answers is exploratory and does not verify any Task 5 F3 improvement.